# 🔍 RAG Saturation & Hallucination Analysis
## Kapita Selekta SC — Final Assignment
### Dataset: IDK-MRC | Model: Qwen2.5-1.5B-Instruct (4-bit) | Retriever: FAISS

**Deskripsi:**  
Notebook ini mengimplementasikan sistem RAG lengkap untuk menginvestigasi:
1. **Saturasi performa** — apakah kualitas jawaban terus meningkat seiring bertambahnya K?
2. **Halusinasi sitasi** — seberapa sering LLM mengutip dokumen yang salah?

**Eksperimen:** K ∈ {1, 3, 5, 10} diuji menggunakan metrik ROUGE, BLEU, dan BERTScore.

## ⚙️ Cell 1 — Install Dependencies

In [ ]:
# ============================================================
# CELL 1: Install semua dependensi yang dibutuhkan
# Runtime: ~2-3 menit pada Google Colab
# ============================================================
!pip install -q datasets faiss-cpu sentence-transformers
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q rouge-score bert-score nltk
!pip install -q matplotlib seaborn pandas tqdm openpyxl


import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ Semua dependensi berhasil diinstall!")


## 📦 Cell 2 — Import Libraries

In [ ]:
# ============================================================
# CELL 2: Import semua library yang diperlukan
# ============================================================
import os
import gc
import re
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm
from collections import defaultdict

import torch
print(f"🖥️  PyTorch version : {torch.__version__}")
print(f"🖥️  CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🖥️  GPU             : {torch.cuda.get_device_name(0)}")
    print(f"🖥️  VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from datasets import load_dataset
import faiss
from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction, corpus_bleu

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("\n✅ Semua library berhasil diimport!")


## 📂 Cell 3 — Load & Eksplorasi Dataset IDK-MRC

In [ ]:
# ============================================================
# CELL 3: Load dataset IDK-MRC dari HuggingFace
# IDK-MRC = Indonesian Knowledge Machine Reading Comprehension
# Format: setiap item memiliki 'context' dan 'qas' (daftar QA)
# ============================================================
print("⏳ Memuat dataset IDK-MRC...")
ds = load_dataset("rifkiaputri/idk-mrc")
print("\n✅ Dataset berhasil dimuat!")
print(f"\n{ds}")

# --- Eksplorasi struktur data ---
print("\n" + "="*60)
print("STRUKTUR DATA (baris pertama split 'train')")
print("="*60)
sample = ds['train'][0]
print(f"\n📝 Context (200 karakter pertama):")
print(f"  {sample['context'][:200]}...")
print(f"\n❓ Jumlah QA pairs pada baris ini: {len(sample['qas'])}")
for i, qa in enumerate(sample['qas']):
    print(f"\n  QA-{i+1}:")
    print(f"    Question   : {qa['question']}")
    print(f"    Is Impossible: {qa['is_impossible']}")
    if qa['answers']:
        print(f"    Answer     : {qa['answers'][0]['text']}")
    else:
        print(f"    Answer     : (tidak ada — pertanyaan tidak bisa dijawab)")

# --- Statistik split ---
print("\n" + "="*60)
print("STATISTIK DATASET")
print("="*60)
for split_name in ['train', 'validation', 'test']:
    split = ds[split_name]
    total_qa = sum(len(row['qas']) for row in split)
    answerable = sum(
        sum(1 for qa in row['qas'] if not qa['is_impossible'])
        for row in split
    )
    print(f"  {split_name:12s}: {len(split):5d} konteks | "
          f"{total_qa:5d} total QA | {answerable:5d} bisa dijawab")


## 🔧 Cell 4 — Preprocessing & Persiapan Data

In [ ]:
# ============================================================
# CELL 4: Preprocessing data
# Kita ekstrak QA pairs yang bisa dijawab (is_impossible=False)
# dari split validation untuk evaluasi.
# ============================================================

def extract_qa_pairs(dataset_split, include_impossible=False):
    """
    Ekstrak QA pairs dari satu split dataset.

    Returns:
        contexts (list): konteks per QA pair
        questions (list): pertanyaan
        answers (list): jawaban ground truth
        context_idx (list): indeks baris asli dalam split
    """
    contexts, questions, answers, ctx_indices = [], [], [], []

    for row_idx, row in enumerate(dataset_split):
        ctx = row['context']
        for qa in row['qas']:
            if qa['is_impossible']:
                if include_impossible:
                    contexts.append(ctx)
                    questions.append(qa['question'])
                    answers.append("")  # tidak ada jawaban
                    ctx_indices.append(row_idx)
                continue
            if not qa['answers']:
                continue
            contexts.append(ctx)
            questions.append(qa['question'])
            answers.append(qa['answers'][0]['text'])
            ctx_indices.append(row_idx)

    return contexts, questions, answers, ctx_indices


# --- Ekstrak dari validation split ---
val_gt_contexts, val_questions, val_answers, val_row_indices =     extract_qa_pairs(ds['validation'], include_impossible=False)

print(f"✅ Total QA pairs bisa dijawab (validation): {len(val_questions)}")
print(f"\nContoh QA pair:")
print(f"  Pertanyaan : {val_questions[0]}")
print(f"  Jawaban    : {val_answers[0]}")
print(f"  Konteks    : {val_gt_contexts[0][:120]}...")

# --- Juga ekstrak train untuk building corpus yang lebih besar ---
train_gt_contexts, train_questions, train_answers, train_row_indices =     extract_qa_pairs(ds['train'], include_impossible=False)

print(f"\n✅ Total QA pairs bisa dijawab (train): {len(train_questions)}")

# --- Statistik panjang teks ---
ctx_lengths  = [len(c.split()) for c in val_gt_contexts[:500]]
q_lengths    = [len(q.split()) for q in val_questions[:500]]
ans_lengths  = [len(a.split()) for a in val_answers[:500]]

print("\n" + "="*60)
print("STATISTIK PANJANG TEKS (validation)")
print("="*60)
print(f"  Konteks  — rata-rata: {np.mean(ctx_lengths):.1f} kata | "
      f"max: {max(ctx_lengths)} kata")
print(f"  Pertanyaan — rata-rata: {np.mean(q_lengths):.1f} kata | "
      f"max: {max(q_lengths)} kata")
print(f"  Jawaban  — rata-rata: {np.mean(ans_lengths):.1f} kata | "
      f"max: {max(ans_lengths)} kata")


## 🗄️ Cell 5 — Bangun Corpus Retrieval

In [ ]:
# ============================================================
# CELL 5: Bangun corpus untuk retrieval
# Corpus = semua konteks unik dari train + validation + test
# Ini mensimulasikan knowledge base yang sesungguhnya
# ============================================================

print("⏳ Membangun corpus dari seluruh split...")

all_corpus_contexts = []
seen = set()

for split_name in ['train', 'validation', 'test']:
    for row in ds[split_name]:
        ctx = row['context'].strip()
        if ctx not in seen:
            seen.add(ctx)
            all_corpus_contexts.append(ctx)

print(f"\n✅ Corpus siap!")
print(f"   Total konteks unik : {len(all_corpus_contexts)}")

# Buat mapping konteks → indeks untuk pencarian cepat
corpus_ctx_to_idx = {ctx: i for i, ctx in enumerate(all_corpus_contexts)}

# Verifikasi: pastikan semua GT contexts ada dalam corpus
missing = sum(
    1 for ctx in val_gt_contexts
    if ctx not in corpus_ctx_to_idx
)
print(f"   GT contexts tidak ditemukan di corpus: {missing} "
      f"(harusnya 0 atau sangat kecil)")

# Contoh corpus
print(f"\nContoh konteks korpus ke-0:")
print(f"  {all_corpus_contexts[0][:200]}...")


## 🧠 Cell 6 — Load Embedding Model (Sentence Transformer)

In [ ]:
# ============================================================
# CELL 6: Load embedding model
# paraphrase-multilingual-MiniLM-L12-v2:
#   - Mendukung 50+ bahasa termasuk Bahasa Indonesia
#   - Ringan (~420 MB) dan cepat
#   - Dimensi embedding: 384
# ============================================================

EMBEDDING_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

print(f"⏳ Memuat embedding model: {EMBEDDING_MODEL_NAME}")
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"✅ Embedding model dimuat!")
print(f"   Dimensi output       : {embed_model.get_sentence_embedding_dimension()}")
print(f"   Max sequence length  : {embed_model.max_seq_length}")

# Test embedding
test_sentence = "Dimana Douwes Dekker dilahirkan?"
test_emb = embed_model.encode([test_sentence])
print(f"\nTest embedding:")
print(f"  Input   : '{test_sentence}'")
print(f"  Shape   : {test_emb.shape}")
print(f"  Norm    : {np.linalg.norm(test_emb):.4f}")


## 🔎 Cell 7 — Bangun FAISS Index

In [ ]:
# ============================================================
# CELL 7: Encode semua corpus contexts & bangun FAISS index
# Menggunakan IndexFlatIP (Inner Product) dengan normalisasi L2
# → efektif = cosine similarity
#
# Estimasi waktu: ~3-5 menit untuk ~4000 konteks di Colab
# ============================================================

EMBED_DIM  = embed_model.get_sentence_embedding_dimension()
BATCH_SIZE = 64

print(f"⏳ Encoding {len(all_corpus_contexts)} konteks...")
print(f"   Batch size : {BATCH_SIZE}")
print(f"   Dimensi    : {EMBED_DIM}")

corpus_embeddings_list = []
for i in tqdm(range(0, len(all_corpus_contexts), BATCH_SIZE),
              desc="Encoding corpus"):
    batch = all_corpus_contexts[i : i + BATCH_SIZE]
    embs  = embed_model.encode(batch,
                               convert_to_numpy=True,
                               show_progress_bar=False,
                               normalize_embeddings=True)  # L2-norm in-place
    corpus_embeddings_list.append(embs)

corpus_embeddings = np.vstack(corpus_embeddings_list).astype('float32')
print(f"\n✅ Encoding selesai! Shape: {corpus_embeddings.shape}")

# --- Bangun FAISS index ---
print("\n⏳ Membangun FAISS index (IndexFlatIP)...")
faiss_index = faiss.IndexFlatIP(EMBED_DIM)
faiss_index.add(corpus_embeddings)

print(f"✅ FAISS index siap!")
print(f"   Total vektor tersimpan : {faiss_index.ntotal}")
print(f"   Tipe index             : {type(faiss_index).__name__}")

# --- Free memory ---
del corpus_embeddings_list
gc.collect()


## 🔍 Cell 8 — Fungsi Retrieval & Uji Coba

In [ ]:
# ============================================================
# CELL 8: Definisi fungsi retrieval dan uji coba
# ============================================================

def retrieve(query: str, k: int = 5):
    """
    Retrieve top-k konteks dari corpus berdasarkan cosine similarity.

    Args:
        query (str): Pertanyaan/query pengguna
        k (int): Jumlah konteks yang diambil

    Returns:
        retrieved_contexts (list[str]): Konteks yang diambil, diurutkan by similarity
        scores (list[float]): Cosine similarity scores (0-1)
        indices (list[int]): Indeks dalam corpus
    """
    # Encode query
    q_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype('float32')

    # Search FAISS
    scores, indices = faiss_index.search(q_emb, k)

    retrieved = [all_corpus_contexts[i] for i in indices[0]]
    return retrieved, scores[0].tolist(), indices[0].tolist()


# --- Uji coba retrieval ---
test_query = val_questions[0]
test_gt    = val_gt_contexts[0]

print(f"🔍 Test Retrieval")
print(f"{'='*60}")
print(f"Pertanyaan : {test_query}")
print(f"Ground Truth Answer: {val_answers[0]}")

for k_test in [1, 3, 5]:
    ret_ctxs, ret_scores, ret_idxs = retrieve(test_query, k=k_test)
    gt_found = any(c == test_gt for c in ret_ctxs)
    print(f"\nK={k_test}: GT ditemukan = {gt_found} | "
          f"Scores: {[f'{s:.3f}' for s in ret_scores]}")
    if k_test == 3:
        for i, (ctx, sc) in enumerate(zip(ret_ctxs, ret_scores)):
            marker = "✅" if ctx == test_gt else "  "
            print(f"  {marker} [{i+1}] (score={sc:.3f}) {ctx[:80]}...")


## 🤖 Cell 9 — Load LLM dengan 4-bit Quantization

In [ ]:
# ============================================================
# CELL 9: Load Qwen2.5-1.5B-Instruct dengan bitsandbytes 4-bit
#
# Model ini dipilih karena:
#   - Hanya ~1.5B parameter → muat di T4 (15 GB VRAM) dengan 4-bit
#   - Instruction-tuned → mengikuti format instruksi dengan baik
#   - Mendukung teks Indonesia cukup baik
#
# Alternatiif jika OOM: google/gemma-2b-it
# ============================================================

LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# LLM_MODEL_NAME = "google/gemma-2b-it"  # ← uncomment jika prefer Gemma

print(f"⏳ Memuat LLM: {LLM_MODEL_NAME}")
print(f"   Konfigurasi: 4-bit NF4 quantization via bitsandbytes")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
llm_model.eval()

# Pastikan pad_token tersedia
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

print(f"\n✅ LLM berhasil dimuat!")
print(f"   Tipe model     : {type(llm_model).__name__}")
print(f"   Device map     : {llm_model.hf_device_map if hasattr(llm_model, 'hf_device_map') else 'auto'}")

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f"   VRAM digunakan : {allocated:.2f} GB allocated / {reserved:.2f} GB reserved")


## 💬 Cell 10 — Prompt Template & Fungsi Generasi

In [ ]:
# ============================================================
# CELL 10: Desain prompt dan fungsi generasi jawaban
#
# Desain prompt RAG:
#  1. Sistem menunjukkan dokumen bernomor [Dokumen 1], [Dokumen 2], ...
#  2. LLM diminta menjawab singkat dan MENYEBUTKAN nomor dokumen
#     yang dijadikan referensi → ini kita gunakan untuk mengukur
#     akurasi sitasi (citation accuracy)
# ============================================================

SYSTEM_PROMPT = (
    "Anda adalah asisten cerdas yang menjawab pertanyaan berdasarkan "
    "dokumen referensi yang diberikan. Jawab secara singkat dan tepat "
    "dalam Bahasa Indonesia. Selalu sebutkan nomor dokumen yang Anda "
    "gunakan sebagai referensi."
)


def build_rag_prompt(question: str, contexts: list[str]) -> str:
    """
    Bangun prompt RAG dengan konteks bernomor untuk mendukung sitasi.

    Format:
        [Dokumen 1]
        <isi konteks 1>

        [Dokumen 2]
        <isi konteks 2>
        ...

        Pertanyaan: <pertanyaan>

        Instruksi:
        1. Jawab berdasarkan dokumen di atas.
        2. Tulis [Referensi: Dokumen X] di akhir jawaban.
    """
    doc_block = ""
    for i, ctx in enumerate(contexts, 1):
        doc_block += f"[Dokumen {i}]\n{ctx}\n\n"

    prompt = (
        f"Berikut adalah dokumen referensi:\n\n"
        f"{doc_block}"
        f"Pertanyaan: {question}\n\n"
        f"Instruksi:\n"
        f"1. Jawab pertanyaan secara singkat dan tepat berdasarkan dokumen di atas.\n"
        f"2. Di akhir jawaban, tambahkan format berikut:\n"
        f"   [Referensi: Dokumen X] — jika menggunakan satu dokumen\n"
        f"   [Referensi: Dokumen X, Dokumen Y] — jika menggunakan lebih dari satu\n"
        f"   [Referensi: Tidak ada] — jika tidak ada dokumen yang relevan\n\n"
        f"Jawaban:"
    )
    return prompt


def generate_answer(question: str,
                    contexts: list[str],
                    max_new_tokens: int = 256) -> str:
    """
    Generate jawaban menggunakan LLM dengan konteks RAG.

    Args:
        question: Pertanyaan yang akan dijawab
        contexts: Daftar konteks hasil retrieval
        max_new_tokens: Maksimum token yang di-generate

    Returns:
        answer (str): Jawaban yang di-generate oleh LLM
    """
    prompt_text = build_rag_prompt(question, contexts)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt_text},
    ]

    # Apply chat template
    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    model_inputs = llm_tokenizer(
        [text],
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(llm_model.device)

    # Generate
    with torch.no_grad():
        output_ids = llm_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,           # greedy decoding → deterministic
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=llm_tokenizer.eos_token_id,
        )

    # Decode hanya bagian baru (after prompt)
    new_token_ids = output_ids[0][model_inputs['input_ids'].shape[1]:]
    answer = llm_tokenizer.decode(new_token_ids, skip_special_tokens=True)

    return answer.strip()


# --- Uji coba generasi ---
print("🧪 Uji coba generasi jawaban (K=3)...")
demo_q   = val_questions[0]
demo_ans = val_answers[0]
demo_ctxs, _, _ = retrieve(demo_q, k=3)

demo_output = generate_answer(demo_q, demo_ctxs)

print(f"\nPertanyaan  : {demo_q}")
print(f"Ground Truth: {demo_ans}")
print(f"\nJawaban LLM :\n{demo_output}")


## 📌 Cell 11 — Ekstraksi Sitasi & Citation Accuracy

In [ ]:
# ============================================================
# CELL 11: Fungsi untuk mengekstrak nomor dokumen yang disitasi
# oleh LLM, dan menghitung citation accuracy
#
# Citation accuracy = persentase kasus di mana LLM menyitasi
# dokumen yang benar-benar mengandung jawaban ground truth
# ============================================================

def extract_cited_doc_numbers(answer_text: str) -> list[int]:
    """
    Ekstrak nomor dokumen yang disitasi dari teks jawaban LLM.
    Menangani berbagai variasi format output.

    Returns:
        list[int]: nomor-nomor dokumen yang disitasi (1-indexed)
    """
    cited = set()

    # Pola 1: [Referensi: Dokumen 1, Dokumen 2]
    pattern1 = r'\[Referensi:\s*Dokumen\s*(\d+(?:[,\s]+Dokumen\s*\d+)*)\]'
    matches1 = re.findall(pattern1, answer_text, re.IGNORECASE)
    for m in matches1:
        nums = re.findall(r'\d+', m)
        cited.update(int(n) for n in nums)

    # Pola 2: [Referensi: 1, 2]
    pattern2 = r'\[Referensi:\s*(\d+(?:[,\s]+\d+)*)\]'
    matches2 = re.findall(pattern2, answer_text, re.IGNORECASE)
    for m in matches2:
        nums = re.findall(r'\d+', m)
        cited.update(int(n) for n in nums)

    # Pola 3: Referensi: Dokumen X (tanpa kurung siku)
    pattern3 = r'Referensi:\s*Dokumen\s*(\d+(?:[,\s]+(?:Dokumen\s*)?\d+)*)'
    matches3 = re.findall(pattern3, answer_text, re.IGNORECASE)
    for m in matches3:
        nums = re.findall(r'\d+', m)
        cited.update(int(n) for n in nums)

    # Pola 4: (Dokumen 1) inline
    pattern4 = r'\(Dokumen\s*(\d+)\)'
    matches4 = re.findall(pattern4, answer_text, re.IGNORECASE)
    cited.update(int(n) for n in matches4)

    # Pola 5: Dokumen 1 tanpa tanda apapun (fallback agresif)
    if not cited:
        pattern5 = r'[Dd]okumen\s*(\d+)'
        matches5 = re.findall(pattern5, answer_text)
        cited.update(int(n) for n in matches5)

    return sorted(list(cited))


def compute_citation_accuracy(
    cited_doc_numbers: list[int],
    ground_truth_context: str,
    retrieved_contexts: list[str]
) -> tuple[int | None, bool]:
    """
    Hitung citation accuracy untuk satu sampel.

    Args:
        cited_doc_numbers : nomor dokumen yang disitasi LLM (1-indexed)
        ground_truth_context : konteks yang benar-benar mengandung jawaban
        retrieved_contexts : daftar konteks yang di-retrieve (urut)

    Returns:
        accuracy (int | None): 1 = benar, 0 = salah, None = GT tidak ada dalam retrieved
        gt_in_retrieved (bool): apakah GT ada dalam retrieved contexts
    """
    # Cari posisi GT dalam retrieved contexts (bisa lebih dari 1 karena duplikat)
    gt_positions = [
        i + 1  # 1-indexed
        for i, ctx in enumerate(retrieved_contexts)
        if ctx == ground_truth_context
    ]

    gt_in_retrieved = len(gt_positions) > 0

    if not gt_in_retrieved:
        # Ground truth tidak ada dalam retrieved → tidak bisa dievaluasi sitasi
        return None, False

    if not cited_doc_numbers:
        # LLM tidak menyitasi apapun → salah
        return 0, True

    # Cek apakah ada nomor yang disitasi cocok dengan posisi GT
    for cited in cited_doc_numbers:
        if cited in gt_positions:
            return 1, True

    return 0, True


# --- Uji coba ---
test_answers_examples = [
    "Dr. Douwes Dekker lahir di Pasuruan. [Referensi: Dokumen 1]",
    "Beliau lahir di Pasuruan, Jawa Timur [Referensi: Dokumen 2, Dokumen 3]",
    "Saya tidak tahu jawabannya. [Referensi: Tidak ada]",
    "Dilahirkan di Pasuruan menurut Dokumen 1.",
]
print("🧪 Uji coba ekstraksi sitasi:")
for ex in test_answers_examples:
    cited = extract_cited_doc_numbers(ex)
    print(f"  Input  : {ex[:70]}")
    print(f"  Sitasi : {cited}\n")


## 📊 Cell 12 — Fungsi Metrik Evaluasi

In [ ]:
# ============================================================
# CELL 12: Implementasi metrik evaluasi
#   - ROUGE-1, ROUGE-2, ROUGE-L : text overlap n-gram
#   - BLEU                       : precision-based n-gram metric
#   - BERTScore                  : semantic similarity (computed later)
# ============================================================

# Inisialisasi ROUGE scorer
rouge_eval = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=False  # jangan stemming → lebih cocok untuk Bahasa Indonesia
)

def compute_rouge_scores(prediction: str, reference: str) -> dict:
    """
    Hitung ROUGE-1, ROUGE-2, ROUGE-L F1 antara prediksi dan referensi.

    Args:
        prediction: Teks yang di-generate oleh model
        reference : Teks ground truth

    Returns:
        dict dengan kunci 'rouge1', 'rouge2', 'rougeL' (float, 0-1)
    """
    if not prediction.strip() or not reference.strip():
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}

    scores = rouge_eval.score(reference, prediction)
    return {
        'rouge1': scores['rouge1'].fmeasure,
        'rouge2': scores['rouge2'].fmeasure,
        'rougeL': scores['rougeL'].fmeasure,
    }


def compute_bleu_score(prediction: str, reference: str) -> float:
    """
    Hitung BLEU score menggunakan smoothing method4 (Chen & Cherry 2014).
    Smoothing diperlukan karena jawaban seringkali sangat pendek.

    Args:
        prediction: Teks yang di-generate
        reference : Teks ground truth

    Returns:
        bleu (float): BLEU score (0-1)
    """
    if not prediction.strip() or not reference.strip():
        return 0.0

    ref_tokens  = reference.lower().split()
    hyp_tokens  = prediction.lower().split()

    if len(hyp_tokens) == 0:
        return 0.0

    try:
        smoothing = SmoothingFunction().method4
        score = sentence_bleu(
            [ref_tokens],
            hyp_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smoothing
        )
    except Exception:
        score = 0.0

    return float(score)


# --- Uji coba metrik ---
pred_ex = "Dr. Douwes Dekker dilahirkan di Pasuruan, Jawa Timur"
ref_ex  = "Pasuruan, Jawa Timur"

rouge_ex = compute_rouge_scores(pred_ex, ref_ex)
bleu_ex  = compute_bleu_score(pred_ex, ref_ex)

print("🧪 Uji coba metrik:")
print(f"  Prediksi  : '{pred_ex}'")
print(f"  Referensi : '{ref_ex}'")
print(f"  ROUGE-1   : {rouge_ex['rouge1']:.4f}")
print(f"  ROUGE-2   : {rouge_ex['rouge2']:.4f}")
print(f"  ROUGE-L   : {rouge_ex['rougeL']:.4f}")
print(f"  BLEU      : {bleu_ex:.4f}")


## 🧪 Cell 13 — Main Experiment Loop (K = 1, 3, 5, 10)

In [ ]:
# ============================================================
# CELL 13: Loop eksperimen utama
#
# Untuk setiap nilai K dalam {1, 3, 5, 10}:
#   1. Retrieve K konteks per pertanyaan
#   2. Generate jawaban dengan LLM
#   3. Hitung ROUGE, BLEU
#   4. Ekstrak sitasi & hitung citation accuracy
#
# NUM_SAMPLES: jumlah sampel yang dievaluasi
#   → 100 sudah cukup representatif dan selesai dalam ~30 menit
#   → Naikkan ke 200-358 untuk hasil lebih akurat (lebih lama)
# ============================================================

K_VALUES    = [1, 3, 5, 10]
NUM_SAMPLES = 100           # ← Bisa dinaikkan untuk hasil lebih akurat

# Sampel acak dari validation set
np.random.seed(42)
sample_idx = np.random.choice(
    len(val_questions),
    size=min(NUM_SAMPLES, len(val_questions)),
    replace=False
)

sampled_questions   = [val_questions[i]    for i in sample_idx]
sampled_answers     = [val_answers[i]      for i in sample_idx]
sampled_gt_contexts = [val_gt_contexts[i]  for i in sample_idx]

print(f"📋 Konfigurasi Eksperimen")
print(f"{'='*50}")
print(f"  K values     : {K_VALUES}")
print(f"  Jumlah sampel: {len(sampled_questions)}")
print(f"  LLM          : {LLM_MODEL_NAME}")
print(f"  Embedding    : {EMBEDDING_MODEL_NAME}")
print(f"  Dataset      : IDK-MRC (validation split)")
print()

# Simpan semua hasil
all_results = {}

TOTAL_RUNS = len(K_VALUES) * len(sampled_questions)
run_count  = 0
start_time = time.time()

for K in K_VALUES:
    print(f"\n{'='*60}")
    print(f"▶ Eksperimen K = {K}")
    print(f"{'='*60}")

    k_data = {
        'K'                 : K,
        'questions'         : [],
        'ground_truths'     : [],
        'generated_answers' : [],
        'rouge1'            : [],
        'rouge2'            : [],
        'rougeL'            : [],
        'bleu'              : [],
        'citation_correct'  : [],  # 1=benar, 0=salah
        'gt_in_retrieved'   : [],  # True/False
        'cited_docs'        : [],  # nomor dokumen yang disitasi
        'n_citation_evaluable': 0, # sampel yang bisa dievaluasi sitasinya
    }

    for i, (q, gt_ans, gt_ctx) in enumerate(
        tqdm(
            zip(sampled_questions, sampled_answers, sampled_gt_contexts),
            total=len(sampled_questions),
            desc=f"K={K}"
        )
    ):
        try:
            # ── 1. Retrieve ──────────────────────────────────────
            ret_ctxs, ret_scores, ret_idxs = retrieve(q, k=K)

            # ── 2. Generate ──────────────────────────────────────
            generated = generate_answer(q, ret_ctxs, max_new_tokens=256)

            # ── 3. ROUGE & BLEU ──────────────────────────────────
            rouge = compute_rouge_scores(generated, gt_ans)
            bleu  = compute_bleu_score(generated, gt_ans)

            # ── 4. Citation accuracy ─────────────────────────────
            cited_nums = extract_cited_doc_numbers(generated)
            cit_acc, gt_in_ret = compute_citation_accuracy(
                cited_nums, gt_ctx, ret_ctxs
            )

            # ── Simpan ───────────────────────────────────────────
            k_data['questions'].append(q)
            k_data['ground_truths'].append(gt_ans)
            k_data['generated_answers'].append(generated)
            k_data['rouge1'].append(rouge['rouge1'])
            k_data['rouge2'].append(rouge['rouge2'])
            k_data['rougeL'].append(rouge['rougeL'])
            k_data['bleu'].append(bleu)
            k_data['gt_in_retrieved'].append(gt_in_ret)
            k_data['cited_docs'].append(cited_nums)

            if cit_acc is not None:
                k_data['citation_correct'].append(cit_acc)
                k_data['n_citation_evaluable'] += 1

        except Exception as e:
            print(f"  ⚠️  Error pada sampel {i}: {e}")
            # Isi dengan nilai default supaya indeks tetap sinkron
            k_data['questions'].append(q)
            k_data['ground_truths'].append(gt_ans)
            k_data['generated_answers'].append("")
            k_data['rouge1'].append(0.0)
            k_data['rouge2'].append(0.0)
            k_data['rougeL'].append(0.0)
            k_data['bleu'].append(0.0)
            k_data['gt_in_retrieved'].append(False)
            k_data['cited_docs'].append([])
            continue

        run_count += 1

    all_results[K] = k_data

    # Print ringkasan sementara
    cit_acc_mean = (
        np.mean(k_data['citation_correct'])
        if k_data['citation_correct'] else float('nan')
    )
    print(f"\n📊 Ringkasan K={K}:")
    print(f"   ROUGE-1  : {np.mean(k_data['rouge1']):.4f}")
    print(f"   ROUGE-2  : {np.mean(k_data['rouge2']):.4f}")
    print(f"   ROUGE-L  : {np.mean(k_data['rougeL']):.4f}")
    print(f"   BLEU     : {np.mean(k_data['bleu']):.4f}")
    print(f"   GT in Retrieved : {np.mean(k_data['gt_in_retrieved'])*100:.1f}%")
    print(f"   Citation Accuracy: {cit_acc_mean:.4f} "
          f"(n={k_data['n_citation_evaluable']})")

elapsed = time.time() - start_time
print(f"\n✅ Eksperimen selesai! Total waktu: {elapsed/60:.1f} menit")


## 🤗 Cell 14 — Hitung BERTScore (Semantic Similarity)

In [ ]:
# ============================================================
# CELL 14: Hitung BERTScore untuk semua K values
#
# BERTScore mengukur semantic similarity menggunakan representasi
# kontekstual dari BERT → lebih robust dari ROUGE/BLEU untuk
# kasus sinonimia atau parafrase.
#
# Model yang digunakan: bert-base-multilingual-cased
# → mendukung Bahasa Indonesia
# ============================================================

print("⏳ Menghitung BERTScore untuk semua K values...")
print("   Model: bert-base-multilingual-cased")
print("   (Proses ini membutuhkan ~2-5 menit)")
print()

bert_model_type = 'bert-base-multilingual-cased'

for K in K_VALUES:
    preds = all_results[K]['generated_answers']
    refs  = all_results[K]['ground_truths']

    # Filter pasangan kosong
    valid_pairs = [
        (p, r) for p, r in zip(preds, refs)
        if p.strip() and r.strip()
    ]

    if not valid_pairs:
        all_results[K]['bertscore_f1'] = [0.0] * len(preds)
        continue

    valid_preds, valid_refs = zip(*valid_pairs)

    print(f"  K={K}: Evaluasi {len(valid_preds)} pasang teks...")

    try:
        P, R, F1 = bert_score_fn(
            list(valid_preds),
            list(valid_refs),
            model_type=bert_model_type,
            lang='id',
            verbose=False,
            batch_size=16,
        )
        f1_scores = F1.tolist()
    except Exception as e:
        print(f"  ⚠️  BERTScore error pada K={K}: {e}")
        f1_scores = [0.0] * len(valid_preds)

    # Kembalikan ke panjang asli (isi 0 untuk yang kosong)
    full_f1 = []
    valid_iter = iter(f1_scores)
    for p, r in zip(preds, refs):
        if p.strip() and r.strip():
            full_f1.append(next(valid_iter))
        else:
            full_f1.append(0.0)

    all_results[K]['bertscore_f1'] = full_f1
    print(f"     BERTScore F1 (avg): {np.mean(f1_scores):.4f}")

print("\n✅ BERTScore selesai!")


## Cell 15 — Tabel Ringkasan 


In [ ]:
# CELL 15: Buat tabel ringkasan semua metrik per nilai K

summary_rows = []
for K in K_VALUES:
    res = all_results[K]

    cit_acc = (
        np.mean(res['citation_correct'])
        if res['citation_correct'] else 0.0
    )
    halluc_rate = 1.0 - cit_acc

    row = {
        'K'                    : K,
        'ROUGE-1'              : round(np.mean(res['rouge1']), 4),
        'ROUGE-2'              : round(np.mean(res['rouge2']), 4),
        'ROUGE-L'              : round(np.mean(res['rougeL']), 4),
        'BLEU'                 : round(np.mean(res['bleu']),   4),
        'BERTScore F1'         : round(np.mean(res.get('bertscore_f1', [0])), 4),
        'Citation Accuracy'    : round(cit_acc,      4),
        'Hallucination Rate'   : round(halluc_rate,  4),
        'GT in Retrieved (%)'  : round(np.mean(res['gt_in_retrieved']) * 100, 2),
        'N Citation Evaluable' : res['n_citation_evaluable'],
    }
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)

print("="*80)
print("TABEL RINGKASAN HASIL EKSPERIMEN RAG")
print("Dataset: IDK-MRC | LLM: Qwen2.5-1.5B-Instruct (4-bit)")
print(f"Jumlah sampel: {NUM_SAMPLES}")
print("="*80)
print(df_summary.to_string(index=False))
print("="*80)

# Hitung delta antar K
print("\n DELTA PERUBAHAN ROUGE-1 ANTAR K:")
r1 = df_summary['ROUGE-1'].tolist()
k_vals = df_summary['K'].tolist()
for i in range(1, len(k_vals)):
    delta = r1[i] - r1[i-1]
    arrow = "↑" if delta > 0 else "↓"
    print(f"  K={k_vals[i-1]} → K={k_vals[i]}: {arrow} {delta:+.4f}")

print("\n DELTA PERUBAHAN HALLUCINATION RATE ANTAR K:")
hr = df_summary['Hallucination Rate'].tolist()
for i in range(1, len(k_vals)):
    delta = hr[i] - hr[i-1]
    arrow = "↑" if delta > 0 else "↓"
    print(f"  K={k_vals[i-1]} → K={k_vals[i]}: {arrow} {delta:+.4f}")

## Cell 16 — Visualisasi Hasil

In [ ]:
# CELL 16: Visualisasi lengkap hasil eksperimen


K_list = df_summary['K'].tolist()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle(
    'Analisis Saturasi & Halusinasi pada Sistem RAG\n'
    'Dataset: IDK-MRC | Model: Qwen2.5-1.5B-Instruct (4-bit) | '
    f'Retriever: FAISS | N={NUM_SAMPLES} sampel',
    fontsize=14, fontweight='bold', y=1.01
)

# ── Plot 1: ROUGE Scores ──────────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(K_list, df_summary['ROUGE-1'], 'o-', label='ROUGE-1',
        color='#2196F3', lw=2.5, ms=9, zorder=3)
ax.plot(K_list, df_summary['ROUGE-2'], 's-', label='ROUGE-2',
        color='#4CAF50', lw=2.5, ms=9, zorder=3)
ax.plot(K_list, df_summary['ROUGE-L'], '^-', label='ROUGE-L',
        color='#F44336', lw=2.5, ms=9, zorder=3)
ax.set_title('ROUGE Scores vs K', fontsize=13, fontweight='bold')
ax.set_xlabel('K (Jumlah Dokumen Retrieved)', fontsize=11)
ax.set_ylabel('Score (F1)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xticks(K_list)
# Annotate max ROUGE-1
best_idx = df_summary['ROUGE-1'].idxmax()
ax.axvline(K_list[best_idx], color='gray', linestyle='--', alpha=0.5,
           label=f'Optimal K={K_list[best_idx]}')
for x, y1, y2, y3 in zip(K_list, df_summary['ROUGE-1'],
                           df_summary['ROUGE-2'], df_summary['ROUGE-L']):
    ax.annotate(f'{y1:.3f}', (x, y1), textcoords="offset points",
                xytext=(0, 8), ha='center', fontsize=8, color='#2196F3')

# ── Plot 2: BLEU Score ────────────────────────────────────────────────────────
ax = axes[0, 1]
bars = ax.bar(K_list, df_summary['BLEU'], color=colors, edgecolor='black',
              linewidth=1.2, alpha=0.85, width=1.8)
ax.set_title('BLEU Score vs K', fontsize=13, fontweight='bold')
ax.set_xlabel('K (Jumlah Dokumen Retrieved)', fontsize=11)
ax.set_ylabel('BLEU Score', fontsize=11)
ax.set_xticks(K_list)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, df_summary['BLEU']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── Plot 3: BERTScore ─────────────────────────────────────────────────────────
ax = axes[0, 2]
ax.plot(K_list, df_summary['BERTScore F1'], 'D-', color='#9C27B0',
        lw=2.5, ms=9, zorder=3)
ax.fill_between(K_list, df_summary['BERTScore F1'],
                alpha=0.15, color='#9C27B0')
ax.set_title('BERTScore F1 (Semantic) vs K', fontsize=13, fontweight='bold')
ax.set_xlabel('K (Jumlah Dokumen Retrieved)', fontsize=11)
ax.set_ylabel('BERTScore F1', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(K_list)
for x, y in zip(K_list, df_summary['BERTScore F1']):
    ax.annotate(f'{y:.3f}', (x, y), textcoords="offset points",
                xytext=(0, 8), ha='center', fontsize=9, color='#9C27B0')

# ── Plot 4: Citation Accuracy ─────────────────────────────────────────────────
ax = axes[1, 0]
bars = ax.bar(K_list, df_summary['Citation Accuracy'],
              color=colors, edgecolor='black', linewidth=1.2,
              alpha=0.85, width=1.8)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.7, label='Threshold 50%')
ax.set_title('Citation Accuracy vs K', fontsize=13, fontweight='bold')
ax.set_xlabel('K (Jumlah Dokumen Retrieved)', fontsize=11)
ax.set_ylabel('Akurasi Sitasi (0–1)', fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_xticks(K_list)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, df_summary['Citation Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── Plot 5: Hallucination Rate ────────────────────────────────────────────────
ax = axes[1, 1]
hall_pct = (df_summary['Hallucination Rate'] * 100).tolist()
bars = ax.bar(K_list, hall_pct,
              color=['#F44336', '#FF9800', '#FFC107', '#8BC34A'],
              edgecolor='black', linewidth=1.2, alpha=0.85, width=1.8)
ax.set_title('Citation Hallucination Rate vs K', fontsize=13, fontweight='bold')
ax.set_xlabel('K (Jumlah Dokumen Retrieved)', fontsize=11)
ax.set_ylabel('Hallucination Rate (%)', fontsize=11)
ax.set_ylim(0, 105)
ax.set_xticks(K_list)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, hall_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── Plot 6: GT Retrieval Recall & Saturation Analysis ─────────────────────────
ax = axes[1, 2]
ax2_twin = ax.twinx()

ax.bar(K_list, df_summary['GT in Retrieved (%)'], color='#607D8B',
       edgecolor='black', alpha=0.6, width=1.8, label='GT Retrieved (%)')
ax2_twin.plot(K_list, df_summary['ROUGE-1'], 'ro-',
              lw=2, ms=8, label='ROUGE-1 (right axis)')

ax.set_title('GT Retrieval Recall & Saturasi ROUGE-1', fontsize=13, fontweight='bold')
ax.set_xlabel('K (Jumlah Dokumen Retrieved)', fontsize=11)
ax.set_ylabel('GT in Retrieved (%)', fontsize=11, color='#455A64')
ax2_twin.set_ylabel('ROUGE-1 Score', fontsize=11, color='red')
ax.set_ylim(0, 110)
ax.set_xticks(K_list)
ax.grid(True, alpha=0.3, axis='y')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='lower right')

for x, y in zip(K_list, df_summary['GT in Retrieved (%)']):
    ax.text(x, y + 1, f'{y:.0f}%', ha='center', fontsize=9, color='#263238')

plt.tight_layout()
plt.savefig('rag_performance_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Grafik disimpan sebagai 'rag_performance_analysis.png'")


## Cell 17 — Distribusi Metrik per K (Box Plot)

In [ ]:
# CELL 17: Box plot distribusi metrik untuk analisis lebih dalam

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Distribusi Metrik per Nilai K (Box Plot)',
             fontsize=14, fontweight='bold')

metrics_to_plot = [
    ('rouge1',       'ROUGE-1 Distribution',  '#2196F3'),
    ('bleu',         'BLEU Distribution',     '#9C27B0'),
    ('bertscore_f1', 'BERTScore F1 Distribution', '#FF9800'),
]

for ax, (metric_key, title, color) in zip(axes, metrics_to_plot):
    data_per_k = [all_results[K].get(metric_key, []) for K in K_VALUES]

    bp = ax.boxplot(
        data_per_k,
        labels=[f'K={k}' for k in K_VALUES],
        patch_artist=True,
        notch=False,
        medianprops={'color': 'red', 'linewidth': 2},
    )

    # Warnai box
    box_colors = ['#BBDEFB', '#90CAF9', '#64B5F6', '#42A5F5']
    for patch, bcolor in zip(bp['boxes'], box_colors):
        patch.set_facecolor(bcolor)
        patch.set_alpha(0.8)

    # Overlay scatter (jitter)
    for i, (data, k) in enumerate(zip(data_per_k, K_VALUES), 1):
        jitter = np.random.normal(0, 0.05, len(data))
        ax.scatter(
            [i + j for j in jitter], data,
            alpha=0.2, s=8, color=color, zorder=2
        )

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')

    # Tambahkan mean line
    means = [np.mean(d) for d in data_per_k]
    ax.plot(range(1, len(K_VALUES)+1), means, 'k--o',
            ms=5, lw=1.5, label='Mean', zorder=5)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('rag_distribution_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Box plot disimpan sebagai 'rag_distribution_boxplot.png'")


## Cell 18 — Analisis Kualitatif (Contoh Jawaban)

In [ ]:
# ============================================================
# CELL 18: Analisis kualitatif — perbandingan jawaban antar K
# Tampilkan beberapa contoh yang menarik:
#   (a) Contoh di mana K besar MEMBANTU
#   (b) Contoh di mana K besar TIDAK membantu (saturasi/noise)
#   (c) Contoh halusinasi sitasi
# ============================================================

pd.set_option('display.max_colwidth', 200)

def show_comparison(idx, title=""):
    print(f"\n{'='*70}")
    print(f"{title} (sampel #{idx})")
    print(f"{'='*70}")
    print(f"Pertanyaan   : {sampled_questions[idx]}")
    print(f"Ground Truth  : {sampled_answers[idx]}")

    for K in K_VALUES:
        gen = all_results[K]['generated_answers'][idx]
        cited = all_results[K]['cited_docs'][idx]
        r1 = all_results[K]['rouge1'][idx]
        print(f"\n  K={K:2d} | ROUGE-1={r1:.3f} | Sitasi={cited}")
        print(f"  └─ {gen[:200]}{'...' if len(gen)>200 else ''}")


# --- Temukan contoh menarik ---

# (a) Kasus terbaik K=5 atau K=10 dibanding K=1
diff_5_vs_1 = [
    all_results[5]['rouge1'][i] - all_results[1]['rouge1'][i]
    for i in range(len(sampled_questions))
]
best_improvement_idx = int(np.argmax(diff_5_vs_1))
show_comparison(best_improvement_idx,
                "K lebih besar MEMBANTU (ROUGE-1 meningkat signifikan)")

# (b) Kasus K=1 lebih baik dari K=10 (noise/saturasi)
diff_1_vs_10 = [
    all_results[1]['rouge1'][i] - all_results[10]['rouge1'][i]
    for i in range(len(sampled_questions))
]
worst_large_k_idx = int(np.argmax(diff_1_vs_10))
show_comparison(worst_large_k_idx,
                "K besar TIDAK membantu (noise/saturasi)")

# (c) Kasus halusinasi sitasi yang jelas
for i in range(len(sampled_questions)):
    if (all_results[5]['gt_in_retrieved'][i] and
        len(all_results[5]['citation_correct']) > i and
        all_results[5]['citation_correct'][i] == 0):
        show_comparison(i, "Contoh HALUSINASI SITASI (GT ada tapi LLM salah sitasi)")
        break

print("\nAnalisis kualitatif selesai!")


## Cell 19 — Analisis Saturasi & Halusinasi (Narasi)

In [ ]:
# CELL 19: Cetak analisis terstruktur untuk dimasukkan ke laporan

print("="*70)
print("ANALISIS SATURASI PERFORMA RAG")
print("="*70)

r1_list  = df_summary['ROUGE-1'].tolist()
r2_list  = df_summary['ROUGE-2'].tolist()
rl_list  = df_summary['ROUGE-L'].tolist()
bl_list  = df_summary['BLEU'].tolist()
bs_list  = df_summary['BERTScore F1'].tolist()
k_list   = df_summary['K'].tolist()
ca_list  = df_summary['Citation Accuracy'].tolist()
hr_list  = df_summary['Hallucination Rate'].tolist()
gt_list  = df_summary['GT in Retrieved (%)'].tolist()

print("\n1. TREN KUALITAS JAWABAN")
print("-"*40)
for i, K in enumerate(k_list):
    print(f"   K={K:2d}: ROUGE-1={r1_list[i]:.4f} | ROUGE-2={r2_list[i]:.4f} | "
          f"ROUGE-L={rl_list[i]:.4f} | BLEU={bl_list[i]:.4f} | "
          f"BERTScore={bs_list[i]:.4f}")

best_k_idx   = int(np.argmax(r1_list))
best_k_val   = k_list[best_k_idx]
print(f"\n   → Nilai K terbaik (ROUGE-1): K={best_k_val}")
print(f"   → Peningkatan K=1→3: {(r1_list[1]-r1_list[0]):+.4f}")
print(f"   → Peningkatan K=3→5: {(r1_list[2]-r1_list[1]):+.4f}")
print(f"   → Peningkatan K=5→10: {(r1_list[3]-r1_list[2]):+.4f}")

if r1_list[1] - r1_list[0] > r1_list[2] - r1_list[1]:
    print("\n   Saturasi terdeteksi: peningkatan ROUGE-1 semakin kecil")
    print(f"      setelah K={k_list[1]}. Penambahan dokumen tidak memberikan")
    print(f"      peningkatan yang proporsional.")

print("\n2. ANALISIS RECALL RETRIEVAL")
print("-"*40)
for i, K in enumerate(k_list):
    print(f"   K={K:2d}: GT dalam retrieved = {gt_list[i]:.1f}%")
print("\n   → Recall meningkat seiring K, namun dokumen tambahan")
print("      seringkali tidak relevan → noise bagi LLM.")

print("\n3. ANALISIS HALUSINASI SITASI")
print("-"*40)
for i, K in enumerate(k_list):
    n_eval = df_summary['N Citation Evaluable'].iloc[i]
    print(f"   K={K:2d}: Citation Accuracy={ca_list[i]:.4f} | "
          f"Hallucination Rate={hr_list[i]*100:.1f}% | "
          f"N evaluable={n_eval}")

best_cit_idx = int(np.argmax(ca_list))
print(f"\n   → Citation accuracy tertinggi: K={k_list[best_cit_idx]}")
print( "   → Hipotesis: Semakin besar K, LLM cenderung bingung menentukan")
print( "     dokumen mana yang relevan, meningkatkan risiko sitasi yang salah.")
print( "   → Fenomena ini disebut 'lost in the middle': LLM cenderung")
print( "     mengabaikan dokumen di posisi tengah pada konteks panjang.")

print("\n" + "="*70)
print("KESIMPULAN")
print("="*70)
print(f"  • Saturasi kualitas mulai terlihat di K>{k_list[best_k_idx]}")
print( "  • Menambah K memperbesar recall tetapi juga menambah noise")
print( "  • Akurasi sitasi cenderung menurun saat K meningkat karena")
print( "    LLM kesulitan mengidentifikasi dokumen yang benar-benar relevan")
print( "  • Trade-off optimal: pilih K yang menyeimbangkan recall dan precision")


## Cell 20 — Simpan Hasil Eksperimen

In [ ]:
# CELL 20: Simpan semua hasil ke file untuk keperluan laporan


import os

OUTPUT_DIR = "rag_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 1. Summary table ---
summary_path = f"{OUTPUT_DIR}/summary_table.csv"
df_summary.to_csv(summary_path, index=False)
print(f"Summary table → {summary_path}")

# --- 2. Detailed per-sample results ---
detail_rows = []
for K in K_VALUES:
    res = all_results[K]
    for i in range(len(res['questions'])):
        detail_rows.append({
            'K'                : K,
            'question'         : res['questions'][i],
            'ground_truth'     : res['ground_truths'][i],
            'generated_answer' : res['generated_answers'][i],
            'rouge1'           : res['rouge1'][i],
            'rouge2'           : res['rouge2'][i],
            'rougeL'           : res['rougeL'][i],
            'bleu'             : res['bleu'][i],
            'bertscore_f1'     : res.get('bertscore_f1', [None]*len(res['questions']))[i],
            'cited_docs'       : str(res['cited_docs'][i]),
            'gt_in_retrieved'  : res['gt_in_retrieved'][i],
        })

df_detail = pd.DataFrame(detail_rows)
detail_path = f"{OUTPUT_DIR}/detailed_results.csv"
df_detail.to_csv(detail_path, index=False)
print(f"Detailed results → {detail_path}")

# --- 3. JSON raw results ---
json_path = f"{OUTPUT_DIR}/raw_results.json"
serializable = {}
for K in K_VALUES:
    serializable[str(K)] = {
        k: v for k, v in all_results[K].items()
        if isinstance(v, (list, int, float, str))
    }
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(serializable, f, ensure_ascii=False, indent=2)
print(f"Raw JSON results → {json_path}")

# --- 4. Salin gambar ---
import shutil
for img in ['rag_performance_analysis.png', 'rag_distribution_boxplot.png']:
    if os.path.exists(img):
        shutil.copy(img, f"{OUTPUT_DIR}/{img}")
        print(f"Grafik → {OUTPUT_DIR}/{img}")

print(f"\Semua file tersimpan di folder: '{OUTPUT_DIR}/'")
print("\nFile yang dihasilkan:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size  = os.path.getsize(fpath)
    print(f"  {f:45s} ({size/1024:.1f} KB)")


## Cell 21 — Download Hasil 

In [ ]:
# ============================================================
# CELL 21: Download file hasil dari Google Colab
# Jalankan cell ini untuk mengunduh semua hasil ke komputer lokal
# ============================================================

try:
    from google.colab import files
    import zipfile

    # Buat ZIP
    zip_path = 'rag_experiment_results.zip'
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for root, dirs, fnames in os.walk(OUTPUT_DIR):
            for fname in fnames:
                fpath = os.path.join(root, fname)
                zf.write(fpath)

    print(f"ZIP file dibuat: {zip_path}")
    files.download(zip_path)
    print("Download dimulai!")

except ImportError:
    print("ℹTidak berjalan di Google Colab.")
    print(f"   File hasil ada di folder: {OUTPUT_DIR}/")
    print("   Gunakan file manager atau SCP untuk mengunduh.")

except Exception as e:
    print(f"Error saat download: {e}")
    print(f"   File tersimpan di: {OUTPUT_DIR}/")
